# Линейная регрессия на NumPy

В ноутбуке реализован полный градиентный спуск для линейной регрессии и выполнено сравнение со `sklearn.linear_model.LinearRegression`.

Для обучения используется один признак и зашумлённая синтетическая выборка.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

## Генерация данных

In [ ]:
np.random.seed(42)

n = 20
d = 1

def y_calc(x, a, b, epsilon=0):
    return a * x + b + epsilon

x_value = np.random.uniform(-5, 6, n).reshape(n, 1)
matrix_X = x_value
noise = np.random.normal(0, 0.5, (n, 1))

y = y_calc(matrix_X, 2, 4)
y_with_noise = y_calc(matrix_X, 2, 4, noise)

## Визуализация данных

In [ ]:
X = matrix_X
target = y_with_noise

plt.figure(figsize=(8, 5))
plt.scatter(X.ravel(), target.ravel(), color='#FF1493', label='целевая переменная с шумом')
plt.scatter(X.ravel(), y.ravel(), color='#1f77b4', label='идеальная зависимость')
plt.xlabel('Признак')
plt.ylabel('Целевая переменная')
plt.title('Синтетические данные')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## Функция потерь и градиенты

In [ ]:
def loss(error, n):
    return np.sum(error ** 2) / (2 * n)

def grad_w(X, error, n):
    return X.T @ error / n

def grad_b(error, n):
    return np.sum(error) / n

## Обучение градиентным спуском

In [ ]:
w = np.zeros((d, 1))
b = 0.0
learning_rate = 0.01
n_iterations = 2000
loss_history = []

initial_error = X @ w + b - target
loss_history.append(loss(initial_error, n))

for _ in range(n_iterations):
    predictions = X @ w + b
    error = predictions - target
    w_gradient = grad_w(X, error, n)
    b_gradient = grad_b(error, n)

    w -= learning_rate * w_gradient
    b -= learning_rate * b_gradient

    predictions = X @ w + b
    error = predictions - target
    loss_history.append(loss(error, n))

print('начальный loss:', loss_history[0])
print('финальный loss:', loss_history[-1])
print('w:', w.ravel())
print('b:', b)

## Убывание функции потерь

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(len(loss_history)), loss_history, color='#FF1493')
plt.yscale('log')
plt.xlabel('Номер итерации')
plt.ylabel('Значение loss')
plt.title('Сходимость градиентного спуска')
plt.grid(True)
plt.tight_layout()
plt.show()

## Сравнение со sklearn

In [ ]:
model = LinearRegression()
model.fit(X, target)

coef_difference = np.abs(model.coef_ - w)
intercept_difference = np.abs(model.intercept_ - b)

print('коэффициент NumPy:', w.ravel())
print('коэффициент sklearn:', model.coef_.ravel())
print('свободный член NumPy:', b)
print('свободный член sklearn:', model.intercept_.ravel())
print('разница коэффициентов:', coef_difference.ravel())
print('разница свободных членов:', intercept_difference.ravel())
print('коэффициенты близки:', np.allclose(model.coef_, w, atol=1e-4, rtol=1e-4))
print('свободные члены близки:', np.allclose(model.intercept_, b, atol=1e-4, rtol=1e-4))

## Вывод

Градиентный спуск уменьшает функцию потерь и приближает параметры модели к решению, найденному `LinearRegression`. Из-за шума параметры не обязаны совпадать с исходными значениями `2` и `4`, но результаты двух методов должны быть близкими.